## White Blood Cell Classification Project using PyTorch with a Stacking / Ensemble model**

### STEP 1 — Install (if needed)

In [1]:
!pip install torch torchvision

### STEP 2 — Import Required Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from PIL import Image

### STEP 3 - Data Import from Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

#### STEP 4 — Device Configuration

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

#### STEP 5 — Load Dataset

In [ ]:
train_path = "/content/drive/MyDrive/white_blood_cells/Train"
test_path = "/content/drive/MyDrive/white_blood_cells/Test-A"


#### STEP 6 — Data Preprocessing & Augmentation

In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ToTensor(),
])

test_transforms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
])

### STEP 7 — Load Dataset

In [ ]:
full_train_dataset = datasets.ImageFolder(train_path, transform=train_transforms)
test_dataset = datasets.ImageFolder(test_path, transform=test_transforms)

train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size

train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

### STEP 8 - Data Visualization

In [ ]:
# labels = [label for _, label in full_train_dataset]

# plt.figure(figsize=(8,5))
# sns.countplot(x=labels)
# plt.title("Class Distribution")
# plt.xlabel("Class Index")
# plt.ylabel("Count")
# plt.show()

### STEP 9 - Sample Images

In [ ]:
# classes = full_train_dataset.classes
# images, labels = next(iter(train_loader))

# plt.figure(figsize=(10,10))
# for i in range(9):
#     plt.subplot(3,3,i+1)
#     img = images[i].permute(1,2,0)
#     plt.imshow(img)
#     plt.title(classes[labels[i]])
#     plt.axis("off")
# plt.show()

### STEP 10 - Compute Class Weights

In [ ]:
num_classes = len(full_train_dataset.classes)

labels = np.array(full_train_dataset.targets)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(num_classes),
    y=labels
)

class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

print("Class Weights:", class_weights)

### STEP 11 — Build Stacking Ensemble Model

In [ ]:
class StackingEnsemble(nn.Module):
    def __init__(self, num_classes):
        super(StackingEnsemble, self).__init__()

        self.model1 = models.resnet50(weights="IMAGENET1K_V1")
        self.model1.fc = nn.Identity()

        self.model2 = models.efficientnet_b0(weights="IMAGENET1K_V1")
        self.model2.classifier = nn.Identity()

        self.model3 = models.mobilenet_v2(weights="IMAGENET1K_V1")
        self.model3.classifier = nn.Identity()

        for param in self.model1.parameters():
            param.requires_grad = False
        for param in self.model2.parameters():
            param.requires_grad = False
        for param in self.model3.parameters():
            param.requires_grad = False

        self.classifier = nn.Sequential(
            nn.Linear(2048 + 1280 + 1280, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        f1 = self.model1(x)
        f2 = self.model2(x)
        f3 = self.model3(x)
        merged = torch.cat((f1, f2, f3), dim=1)
        return self.classifier(merged)

### STEP 12 — Initialize Model

In [ ]:
model = StackingEnsemble(num_classes).to(device)

### STEP 13 — Loss & Optimizer

In [ ]:
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=1e-4)

### STEP 14 — Training Loop with Validation Tracking

In [ ]:
num_epochs = 15

train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(num_epochs):

    model.train()
    running_loss, correct, total = 0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (preds == labels).sum().item()

    train_loss = running_loss / len(train_loader)
    train_acc = correct / total

    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (preds == labels).sum().item()

    val_loss /= len(val_loader)
    val_acc = val_correct / val_total

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(f"Epoch {epoch+1}/{num_epochs} "
          f"Train Loss: {train_loss:.4f} "
          f"Val Loss: {val_loss:.4f} "
          f"Train Acc: {train_acc:.4f} "
          f"Val Acc: {val_acc:.4f}")

#### LOSS

In [ ]:
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")
plt.legend()
plt.title("Loss Curve")
plt.show()

### Accuracy Curve

In [ ]:
plt.plot(train_accs, label="Train Accuracy")
plt.plot(val_accs, label="Validation Accuracy")
plt.legend()
plt.title("Accuracy Curve")
plt.show()

### STEP 16 - Model Evaluation

In [ ]:
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

print(confusion_matrix(all_labels, all_preds))
print(classification_report(all_labels, all_preds, target_names=full_train_dataset.classes))

### STEP 17 - Confusion Matrix Plot

In [ ]:
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d',
            xticklabels=full_train_dataset.classes,
            yticklabels=full_train_dataset.classes)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

### STEP 18 - FINAL RESULT VISUALIZATION — Prediction Results Grid

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def show_prediction_results(model, test_loader, class_names, num_images=9):
    model.eval()
    images_shown = 0

    plt.figure(figsize=(12,12))

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            for i in range(images.size(0)):
                if images_shown >= num_images:
                    break

                img = images[i].cpu().permute(1,2,0).numpy()
                true_label = class_names[labels[i]]
                predicted_label = class_names[preds[i]]

                plt.subplot(3,3,images_shown+1)
                plt.imshow(img)

                if true_label == predicted_label:
                    color = "green"
                else:
                    color = "red"

                plt.title(f"True: {true_label}\nPred: {predicted_label}", color=color)
                plt.axis("off")

                images_shown += 1

            if images_shown >= num_images:
                break

    plt.tight_layout()
    plt.show()

In [ ]:
show_results(model, test_loader, full_train_dataset.classes)

### STEP 18 - Run the Final Visualization

In [ ]:
show_prediction_results(
    model,
    test_loader,
    full_train_dataset.classes,
    num_images=9
)

### STEP 19 — Save Model

In [ ]:
torch.save(model.state_dict(), "WBC_Stacking_Ensemble_PyTorch.pth")